<a href="https://colab.research.google.com/github/VladzioG/Classifier-of-Military-Sounds/blob/main/data_processing_and_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pathlib import Path
import pandas as pd

ROOT_DIRS = [Path("processed"), Path("processed2")]
CLASSES = ["weapons", "air vehicles", "tanks"]

data = []

for root_dir in ROOT_DIRS:
    for target_class in CLASSES:
        class_folder = root_dir / target_class

        if class_folder.exists():

            for wav_file in class_folder.glob("*.wav"):
                data.append({
                    "file_path": str(wav_file),
                    "label": target_class,
                    "source_batch": root_dir.name
                })

df = pd.DataFrame(data)

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Загалом знайдено файлів: {len(df)}")
print("\nРозподіл по класах:")
print(df["label"].value_counts())

In [ ]:
df.head()

# EDA та препроцесинг аудіо


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Візуалізація балансу класів
plt.figure(figsize=(8, 4))
sns.countplot(data=df, x='label', order=['weapons', 'air vehicles', 'tanks'])
plt.title('Розподіл файлів за класами')
plt.xlabel('Клас')
plt.ylabel('Кількість')
plt.show()

In [ ]:
from scipy.io import wavfile
import numpy as np
from scipy import signal

def extract_audio_meta(file_path):
    sr, audio = wavfile.read(file_path)
    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)
    audio = audio.astype(np.float32)
    peak = np.max(np.abs(audio))
    if peak > 0:
        audio = audio / peak
    duration = len(audio) / sr
    channels = 1
    return pd.Series([duration, sr, channels])

df[["duration", "sample_rate", "channels"]] = df["file_path"].apply(extract_audio_meta)


In [ ]:
df.describe()

In [ ]:
from scipy.io import wavfile

sr, y = wavfile.read(df["file_path"].iloc[0])
if y.ndim > 1:
    y = np.mean(y, axis=1)
y = y.astype(np.float32)
if np.max(np.abs(y)) > 0:
    y = y / np.max(np.abs(y))

plt.figure(figsize=(10, 3))
plt.plot(np.linspace(0, len(y) / sr, len(y)), y, color="#1f77b4")
plt.title(f"Форма хвилі для класу: {df['label'].iloc[0]}")
plt.xlabel("Час, сек")
plt.ylabel("Нормалізована амплітуда")
plt.tight_layout()
plt.show()


In [ ]:
from scipy import signal
freqs, times, Sxx = signal.spectrogram(y, fs=sr, window="hann", nperseg=1024, noverlap=512)
Sxx_db = 10 * np.log10(Sxx + 1e-10)

plt.figure(figsize=(10, 4))
plt.pcolormesh(times, freqs, Sxx_db, shading="gouraud", cmap="viridis")
plt.colorbar(label="Power (dB)")
plt.title("Спектрограма сигналу")
plt.ylabel("Частота, Гц")
plt.xlabel("Час, сек")
plt.ylim(0, sr / 2)
plt.tight_layout()
plt.show()


In [ ]:
import librosa
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from tqdm import tqdm

tqdm.pandas(desc="Feature Extraction")

def preprocess_and_extract(file_path, target_sr=22050, duration=5.0, top_db=20):
    try:
        y, sr = librosa.load(file_path, sr=target_sr)
        y, _ = librosa.effects.trim(y, top_db=top_db)

        target_length = int(target_sr * duration)
        if len(y) > target_length:
            y = y[:target_length]
        else:
            y = np.pad(y, (0, max(0, target_length - len(y))), "constant")

        features = []

        # Витягування MFCC (13 коефіцієнтів)
        mfcc = librosa.feature.mfcc(y=y, sr=target_sr, n_mfcc=13)
        features.extend(np.mean(mfcc, axis=1))
        features.extend(np.std(mfcc, axis=1))

        # Витягування Spectral Centroid
        centroid = librosa.feature.spectral_centroid(y=y, sr=target_sr)
        features.append(np.mean(centroid))
        features.append(np.std(centroid))

        # Витягування Zero Crossing Rate
        zcr = librosa.feature.zero_crossing_rate(y=y)
        features.append(np.mean(zcr))
        features.append(np.std(zcr))

        return np.array(features)
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

In [ ]:
df['features'] = df['file_path'].progress_apply(preprocess_and_extract)
df = df.dropna(subset=['features'])

X_raw = np.stack(df['features'].values)
y_labels = df['label'].values

print(f"Features shape: {X_raw.shape}")

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

In [ ]:
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_2d = tsne.fit_transform(X_scaled)

df['tsne_1'] = X_2d[:, 0]
df['tsne_2'] = X_2d[:, 1]

sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 8))

sns.scatterplot(
    data=df,
    x='tsne_1',
    y='tsne_2',
    hue='label',
    palette='Set1',
    alpha=0.7,
    s=50
)

plt.title('Audio Feature Visualization via t-SNE', fontsize=16)
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')
plt.legend(title='Class', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_raw, y_labels, test_size=0.2, random_state=42,stratify=y_labels)
print(f"Train set size: {X_train.shape} samples")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42))
])

param_grid_lr = {
    'classifier__C': [0.01, 0.1, 1.0, 10.0, 100.0],
    'classifier__solver': ['lbfgs', 'saga']
}

grid_lr = GridSearchCV(pipe_lr,
                        param_grid_lr,
                        cv=5, scoring='f1_macro',
                        n_jobs=-1)

grid_lr.fit(X_train, y_train)
print('Best Logistic Regression Params:', grid_lr.best_params_)

In [ ]:
from sklearn.metrics import f1_score, recall_score,precision_score,accuracy_score
result = []
def evaluate_model(model_name,model, X_test, y_test, average='macro'):
    y_pred = model.predict(X_test)
    metrics = {
        'Accuracy': accuracy_score(y_true=y_test,y_pred=y_pred),
        "precision": precision_score(y_test, y_pred, average=average),
        "recall": recall_score(y_test, y_pred, average=average),
        "f1": f1_score(y_test, y_pred, average=average)
    }
    result.append(metrics)
    print(f'Оцінка моделі {model_name} :')
    return metrics
print('Метрики:',evaluate_model('Log_reg',model=grid_lr.best_estimator_,X_test=X_test,y_test=y_test))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_estimator(grid_lr.best_estimator_, X_test, y_test, display_labels=grid_lr.classes_, cmap='Blues')
plt.title('Confusion Matrix - Logistic Regression')
plt.show()

In [ ]:
from sklearn.svm import SVC
pipe_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', SVC(class_weight='balanced', probability=True, random_state=42))
])

param_grid_svm = {
    'classifier__C': [0.1, 1.0, 10.0, 100.0],
    'classifier__gamma': ['scale', 'auto', 0.01, 0.1],
    'classifier__kernel': ['rbf', 'linear']
}

grid_svm = GridSearchCV(pipe_svm, param_grid_svm, cv=5, scoring='f1_macro', n_jobs=-1)
grid_svm.fit(X_train, y_train)

best_model = grid_svm.best_estimator_
y_score = best_model.predict_proba(X_test)

In [ ]:
print('Метрики:',evaluate_model('SVM',model=grid_svm.best_estimator_,X_test=X_test,y_test=y_test))

In [ ]:
ConfusionMatrixDisplay.from_estimator(grid_svm.best_estimator_, X_test, y_test, display_labels=grid_svm.classes_, cmap='Blues')
plt.title('Confusion Matrix - Logistic Regression')
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

y_test_bin = label_binarize(y_test, classes=np.unique(y_labels))
n_classes = y_test_bin.shape[1]

best_model = grid_svm.best_estimator_

y_score = best_model.predict_proba(X_test)

plt.figure(figsize=(10, 8))
colors = ['aqua', 'darkorange', 'cornflowerblue']
classes_names = np.unique(y_labels)

for i, color in zip(range(n_classes), colors):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, lw=2,
             label=f'ROC curve of class {classes_names[i]} (area = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multi-class ROC Curve (One-vs-Rest)')
plt.legend(loc="lower right")
plt.show()

In [ ]:
import joblib
joblib.dump(grid_svm.best_estimator_, 'audio_classifier_model.pkl')
joblib.dump(scaler, 'audio_scaler.pkl')
joblib.dump(np.unique(y_labels), 'classes.pkl')